# Lab 7 — Tune the Champion, Close the Loop
**SDA-AIE-111 · Day 4 · Hour 5 · 50 minutes · shared with capstone kickoff**

**Objective.** Budgeted randomised search over the full champion pipeline; read the results beyond the winner; execute the final refit + **single** test-set evaluation; deliver the model card.

⏱ The search runs ~10 minutes. Start it, then draft your model card skeleton while it runs — that is deliberate time design, and exactly when you should read the capstone briefs.


In [1]:
# --- Course setup (identical in every lab) -----------------------------------
import sys, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
RANDOM_STATE = 42

def find_repo_root() -> Path:
    """Locate the course repo root by looking for data/manafeth_customers.parquet."""
    for cand in [Path.cwd(), *Path.cwd().parents]:
        if (cand / "data" / "manafeth_customers.parquet").exists():
            return cand
    raise FileNotFoundError(
        "Course data not found — copy the Manafeth data package files into <repo>/data/")

ROOT = find_repo_root()
DATA = ROOT / "data"
sys.path.insert(0, str(ROOT / "src"))
print("repo root:", ROOT)

repo root: /home/claude/amlf/repo


In [2]:
from sklearn.model_selection import train_test_split
from manafeth.features import (build_preprocessor, build_feature_table,
                               MODEL_NUM_COLS, CAT_COLS, LEAKY)
from manafeth.evaluation import CV, evaluate, recall_at_budget, precision_at_budget

customers = pd.read_parquet(DATA / "manafeth_customers.parquet")
orders = pd.read_parquet(DATA / "manafeth_orders.parquet")
full = build_feature_table(customers, orders, pd.Timestamp("2025-11-01"))

X_all = full.drop(columns=["churned_30d", "customer_id", *LEAKY])
y_all = full["churned_30d"]
X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all, test_size=0.20, stratify=y_all, random_state=RANDOM_STATE)
print("train:", X_train.shape, "| churn:", round(y_train.mean(), 3))

train: (38400, 22) | churn: 0.14


## Task 1 — Pre-tuning checklist *(5 min)*

Fill this in **before** launching the search:

- [ ] Learning-curve verdict (Lab 4): more data won't help → tuning is a legitimate next lever
- [ ] Error-analysis debts (Lab 4 findings) noted — tuning fixes neither of them
- [ ] Leakage tests green: shuffled-label PR-AUC ≈ 0.14 (Lab 3)
- [ ] **Budget statement: 30 draws × 5 folds = 150 fits, decided now, not extended after peeking**


## Task 2 — Launch the search *(15 min, ~10 of them compute)*

In [3]:
from scipy.stats import loguniform, randint, uniform
from sklearn.model_selection import RandomizedSearchCV
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier

pipe = Pipeline([
    ("prep", build_preprocessor(MODEL_NUM_COLS, CAT_COLS)),
    ("clf", XGBClassifier(eval_metric="aucpr", random_state=RANDOM_STATE, n_jobs=-1)),
])

space = {
    # log-uniform where ratios matter, integers where structure does
    "clf__learning_rate":    loguniform(0.01, 0.3),
    "clf__n_estimators":     randint(200, 1200),
    "clf__max_depth":        randint(3, 8),
    "clf__min_child_weight": randint(1, 8),
    "clf__subsample":        uniform(0.6, 0.4),          # [0.6, 1.0]
    "clf__colsample_bytree": uniform(0.6, 0.4),
    "clf__reg_lambda":       loguniform(0.1, 10),
    "clf__scale_pos_weight": uniform(4, 5),              # around the 6.1 imbalance ratio
    # preprocessing is searchable too — one honest example:
    "prep__num__impute__strategy": ["median", "mean"],
}

search = RandomizedSearchCV(
    pipe, space,
    n_iter=30,                                           # the budget, decided BEFORE running
    cv=CV, scoring="average_precision",
    n_jobs=-1, random_state=RANDOM_STATE, refit=True, verbose=1,
)
search.fit(X_train, y_train)
print("best CV PR-AUC:", round(search.best_score_, 3), " <- a SELECTION score, not a promise")
print({k: (round(v, 4) if isinstance(v, float) else v)
       for k, v in search.best_params_.items()})

Fitting 5 folds for each of 30 candidates, totalling 150 fits


best CV PR-AUC: 0.58  <- a SELECTION score, not a promise
{'clf__colsample_bytree': np.float64(0.7272), 'clf__learning_rate': np.float64(0.0145), 'clf__max_depth': 3, 'clf__min_child_weight': 3, 'clf__n_estimators': 1157, 'clf__reg_lambda': np.float64(0.1002), 'clf__scale_pos_weight': np.float64(5.7628), 'clf__subsample': np.float64(0.7219), 'prep__num__impute__strategy': 'mean'}


## Task 3 — Read the search beyond the winner *(10 min)*

In [4]:
res = (pd.DataFrame(search.cv_results_)
         .sort_values("rank_test_score")
         [["mean_test_score", "std_test_score", "param_clf__learning_rate",
           "param_clf__max_depth", "param_clf__n_estimators"]]
         .head(10).round(4))
print(res.to_string())
spread = res["mean_test_score"].max() - res["mean_test_score"].min()
print(f"\nTop-10 spread: {spread:.4f}",
      "-> flat optimum: the choice is robust, stop searching." if spread < 0.01
      else "-> the optimum has structure; the NEXT search (not this one) could narrow.")

full_res = pd.DataFrame(search.cv_results_)
for p in ["param_clf__learning_rate", "param_clf__max_depth", "param_clf__subsample"]:
    corr = full_res[p].astype(float).corr(full_res["mean_test_score"], method="spearman")
    print(f"{p:32s} spearman {corr:+.2f}")
print("One sentence: which knob would the next search narrow, and which would it drop?")

    mean_test_score  std_test_score  param_clf__learning_rate  param_clf__max_depth  param_clf__n_estimators
16           0.5799          0.0093                    0.0145                     3                     1157
28           0.5796          0.0089                    0.0214                     3                     1000
10           0.5791          0.0096                    0.0148                     3                     1129
19           0.5768          0.0106                    0.0263                     3                      312
24           0.5765          0.0082                    0.0147                     4                      462
1            0.5765          0.0080                    0.0163                     5                      508
20           0.5744          0.0077                    0.0226                     4                      883
4            0.5742          0.0066                    0.0125                     6                      515
27           0.5739

## Task 4 — The final act: the quarantine ends. Once. *(10 min)*

The test partition saved in Lab 1 is opened now, for the first and only time. If the test score sits within ~1 std of the CV estimate, the workflow kept its promise. If the gap exceeds 3 stds — stop and investigate (stale kernel? leak amplified by tuning?) before narrating anything.


In [5]:
from sklearn.metrics import average_precision_score

final_model = search.best_estimator_          # refit=True: already trained on ALL training data

# X_test/y_test come from the SAME seeded split as Lab 1 — verify the quarantine hash
quarantine = pd.read_parquet(DATA / "splits" / "test_quarantine.parquet")
assert len(quarantine) == len(X_test), "test partition mismatch — investigate before continuing"

test_proba = final_model.predict_proba(X_test)[:, 1]
test_pr_auc = average_precision_score(y_test, test_proba)
cv_mean, cv_std = search.best_score_, pd.DataFrame(search.cv_results_).loc[
    search.best_index_, "std_test_score"]
print(f"CV estimate {cv_mean:.3f} ± {cv_std:.3f}  |  test (once): {test_pr_auc:.3f}")
gap = abs(test_pr_auc - cv_mean) / cv_std
print(f"agreement check: |gap| = {gap:.1f} std ->",
      "the workflow kept its promise ✔" if gap <= 3 else "STOP — investigate before reporting")

print(f"\nOperating point at the voucher budget (top-20% of test scores):")
print(f"recall@20%: {recall_at_budget(y_test, test_proba):.2f}   "
      f"precision@20%: {precision_at_budget(y_test, test_proba):.2f}")

CV estimate 0.580 ± 0.009  |  test (once): 0.567
agreement check: |gap| = 1.4 std -> the workflow kept its promise ✔

Operating point at the voucher budget (top-20% of test scores):
recall@20%: 0.66   precision@20%: 0.46


## Task 5 — The model card *(10 min)*

Complete `project/MODEL_CARD.md` from the template. The card takes the **test** number; `best_score_` is labelled as a selection score. Sections: intended use · data lineage · features + leakage attestation · performance (CV ± std, single test evaluation, operating point) · known weaknesses (your Lab 4 findings) · retraining cadence & owner.

**Commit:** `feat(lab7): tuned champion, test evaluation, model card` — **the golden thread is complete.**

> The sentence to say out loud when the agreement check passes: *the quarantine held, the estimate was honest, the workflow kept its promise.*
